# DA2-MODEL-04 — TruncatedSVD: Giảm chiều dữ liệu

> **Yêu cầu:** Chạy `DA2-MODEL-01` trước

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

X_train_sc = np.load('/home/jovyan/work/sv4/shared/X_train_sc.npy')
X_test_sc  = np.load('/home/jovyan/work/sv4/shared/X_test_sc.npy')
y_train    = np.load('/home/jovyan/work/sv4/shared/y_train.npy')
y_test     = np.load('/home/jovyan/work/sv4/shared/y_test.npy')
y_pred_rf  = np.load('/home/jovyan/work/sv4/shared/y_pred_rf.npy')

X_all_sc = np.vstack([X_train_sc, X_test_sc])
print(f'Toàn bộ dữ liệu: {X_all_sc.shape}')

In [ ]:
svd = TruncatedSVD(n_components=5, random_state=42)
X_svd = svd.fit_transform(X_all_sc)

print('Kích thước trước SVD:', X_all_sc.shape)
print('Kích thước sau  SVD:', X_svd.shape)
print()
print('Explained Variance Ratio:')
for i, var in enumerate(svd.explained_variance_ratio_):
    print(f'  Component {i+1}: {var:.4f} ({var*100:.2f}%)')
print(f'  Tổng cộng     : {svd.explained_variance_ratio_.sum():.4f} ({svd.explained_variance_ratio_.sum()*100:.2f}%)')

np.save('/home/jovyan/work/sv4/shared/X_svd.npy', X_svd)
with open('/home/jovyan/work/sv4/shared/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)
print('Đã lưu X_svd và svd_model vào shared/')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 6), svd.explained_variance_ratio_, color='steelblue')
axes[0].set_title('Explained Variance mỗi Component')
axes[0].set_xlabel('SVD Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_xticks(range(1, 6))

cumsum = np.cumsum(svd.explained_variance_ratio_)
axes[1].plot(range(1, 6), cumsum, marker='o', color='orange')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Số Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].set_xticks(range(1, 6))
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/jovyan/work/sv4/DA2-MODEL-04/SVD_variance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# So sánh RF trước vs sau SVD
split = len(X_train_sc)
rf_svd = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_svd.fit(X_svd[:split], y_train)
y_pred_svd = rf_svd.predict(X_svd[split:])

import pandas as pd
comp = pd.DataFrame({
    'Tập dữ liệu': ['Gốc (7 features)', 'Sau SVD (5 components)'],
    'Accuracy'   : [accuracy_score(y_test, y_pred_rf),  accuracy_score(y_test, y_pred_svd)],
    'F1-Score'   : [f1_score(y_test, y_pred_rf),        f1_score(y_test, y_pred_svd)],
}).set_index('Tập dữ liệu').round(4)
print('So sánh RF trước và sau SVD:')
print(comp.to_string())